## Build In Tools

In [ ]:
!pip install -U ddgs

In [ ]:
! pip install langchain-experimental 

In [6]:
from langchain_community.tools import DuckDuckGoSearchRun ,ShellTool
# from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)


#tool binding
llm_with_tools = llm.bind_tools([DuckDuckGoSearchRun(), ShellTool()])



In [23]:
res = llm_with_tools.invoke("search for 'What is LangChain?' using DuckDuckGoSearchRun tool.")

In [24]:
res

AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the duckduckgo_search function.', 'tool_calls': [{'id': 'fc_70c560d1-ce44-4646-afac-b23062c0611f', 'function': {'arguments': '{"query":"What is LangChain?"}', 'name': 'duckduckgo_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 191, 'total_tokens': 233, 'completion_time': 0.049533418, 'prompt_time': 0.023156596, 'queue_time': 0.443337912, 'total_time': 0.072690014, 'completion_tokens_details': {'reasoning_tokens': 12}}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e189667b30', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--b46d34d1-fc46-4e1d-99cc-7837ab799434-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'What is LangChain?'}, 'id': 'fc_70c560d1-ce44-4646-afac-b23062c0611f', 'type': 'tool_call'}], usage_metadata={'input_tokens': 191, 'output_toke

In [25]:
res.additional_kwargs

{'reasoning_content': 'We need to use the duckduckgo_search function.',
 'tool_calls': [{'id': 'fc_70c560d1-ce44-4646-afac-b23062c0611f',
   'function': {'arguments': '{"query":"What is LangChain?"}',
    'name': 'duckduckgo_search'},
   'type': 'function'}]}

In [26]:
res.tool_calls[0]

{'name': 'duckduckgo_search',
 'args': {'query': 'What is LangChain?'},
 'id': 'fc_70c560d1-ce44-4646-afac-b23062c0611f',
 'type': 'tool_call'}

In [27]:
DuckDuckGoSearchRun().invoke(res.tool_calls[0]['args'])

"LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases … Aug 25, 2025 · LangChain is an open-source framework designed to simplify the creation of applications using large language models (LLMs). It provides a standard interface for … LangChain is the easiest way to start building agents and applications powered by LLMs. With under 10 lines of code, you can connect to OpenAI, Anthropic, Google, and more. LangChain is an open-source orchestration framework that simplifies building applications with large language models (LLMs). It provides tools and components to connect LLMs with … 5 days ago · LangChain focuses on building sequences of steps called chains, while LangGraph takes things a step further by adding memory, branching, and feedback loops to make your AI …"

#### Tool execution


In [40]:
res = llm_with_tools.invoke("""use shell to show" python --version"  output. and then""")

In [41]:
res.tool_calls

[{'name': 'terminal',
  'args': {'commands': 'python --version'},
  'id': 'fc_df5c5b6d-c0c9-4015-8f91-137e77a3cfa6',
  'type': 'tool_call'}]

In [43]:
ShellTool().invoke(res.tool_calls[0]['args']).strip()

Executing command:
 ['python --version']


e:\Langchain\venv\Lib\site-packages\langchain_community\tools\shell\tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


'Python 3.11.0'

## Custom tools

In [44]:
from langchain_core.tools import tool

@tool
def add(a:float,b:float) -> int:
    '''Adds two numbers together.'''
    return a + b

@tool
def multiply(a:float,b:float) -> float:
    '''Multiplies two numbers together.'''
    return a * b

@tool
def sub(a:float,b:float) -> int:
    '''Subtracts two numbers.'''
    return a - b


In [ ]:
from langchain_groq import ChatGroq
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

llm_with_custom_tools = model.bind_tools([add, multiply, sub])

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser,JsonOutputParser
prompt = PromptTemplate(
    template="Use the following tools to answer the question below:\n\n {query}"
)

from langchain_core.runnables import RunnableLambda

import json
def extract_tool_calls(ai_msg):
    tool_calls = ai_msg.tool_calls or []
    return json.dumps(tool_calls) 



chain = prompt | llm_with_custom_tools | RunnableLambda(extract_tool_calls)  | JsonOutputParser()

In [48]:
tooltoinvoke = chain.invoke({"query": "What is 10 plus 5 and then multiplied by 3"})

In [49]:
add.invoke(tooltoinvoke[0]['args'])

15.0